### Задание 1. Функция vs процедура в обработке ошибок

1. Процедура `log_error(path: str, reason: str) -> None` — логирует ошибку через `logging.warning`.
2. Функция `parse_int_safe(value: str) -> int | None` — возвращает число или `None` при `ValueError`.

In [3]:
import logging

logger = logging.getLogger("task1")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")


def log_error(path: str, reason: str) -> None:
    """Процедура: логирует ошибку и ничего не возвращает."""
    logger.warning("Ошибка в %s: %s", path, reason)


def parse_int_safe(value: str) -> int | None:
    """Функция: возвращает int или None при ValueError."""
    try:
        return int(value)
    except ValueError:
        return None


log_error("data.json", "не найден")
print(parse_int_safe("42"))    # 42
print(parse_int_safe("abc"))   # None

WARNING task1: Ошибка в data.json: не найден


42
None


### Задание 2. Все виды исключений

Для каждого выражения предскажите тип исключения, затем проверьте в try/except. Покажите `__mro__` и объясните, от кого наследуется.

```python
1 / 0
int("abc")
[1, 2][5]
{"a": 1}["b"]
"5" + 5
open("no_such_file.txt")
```

In [4]:
expressions = [
    ("1 / 0", lambda: 1 / 0),
    ("int('abc')", lambda: int("abc")),
    ("[1, 2][5]", lambda: [1, 2][5]),
    ("{'a': 1}['b']", lambda: {"a": 1}["b"]),
    ("'5' + 5", lambda: "5" + 5),
    ("open('no_such_file.txt')", lambda: open("no_such_file.txt")),
]

for label, func in expressions:
    try:
        func()
    except Exception as exc:
        print(f"{label:30s} -> {type(exc).__name__}")
        print("    MRO:", [c.__name__ for c in type(exc).__mro__])

1 / 0                          -> ZeroDivisionError
    MRO: ['ZeroDivisionError', 'ArithmeticError', 'Exception', 'BaseException', 'object']
int('abc')                     -> ValueError
    MRO: ['ValueError', 'Exception', 'BaseException', 'object']
[1, 2][5]                      -> IndexError
    MRO: ['IndexError', 'LookupError', 'Exception', 'BaseException', 'object']
{'a': 1}['b']                  -> KeyError
    MRO: ['KeyError', 'LookupError', 'Exception', 'BaseException', 'object']
'5' + 5                        -> TypeError
    MRO: ['TypeError', 'Exception', 'BaseException', 'object']
open('no_such_file.txt')       -> FileNotFoundError
    MRO: ['FileNotFoundError', 'OSError', 'Exception', 'BaseException', 'object']


### Задание 3. try / except / else / finally

Реализуйте `safe_divide(a, b) -> float | None`:
- в `try` — деление;
- в `except ZeroDivisionError` — лог и `None`;
- в `else` — сообщение об успехе;
- в `finally` — сообщение «операция завершена».

Продемонстрируйте: `(10, 2)`, `(10, 0)`, `(10, "x")`.

In [7]:
def safe_divide(a, b) -> float | None:
    try:
        result = a / b
    except ZeroDivisionError as exc:
        logger.warning("Деление на ноль: %s", exc)
        return None
    except TypeError as exc:
        logger.warning("Неверный тип: %s", exc)
        return None
    else:
        logger.info("Успешно: %s / %s = %s", a, b, result)
        return result
    finally:
        logger.info("Операция завершена")


print(safe_divide(10, 2))    # 5.0
print(safe_divide(10, 0))    # None
print(safe_divide(10, "x"))  # None

INFO task1: Успешно: 10 / 2 = 5.0
INFO task1: Операция завершена
WARNING task1: Деление на ноль: division by zero
INFO task1: Операция завершена
WARNING task1: Неверный тип: unsupported operand type(s) for /: 'int' and 'str'
INFO task1: Операция завершена


5.0
None
None


### Задание 4. Своя иерархия исключений

Создайте `IngestError`, `ConfigError`, `DocumentError`. Продемонстрируйте, что `except IngestError` ловит обоих наследников.

In [8]:
class IngestError(Exception):
    """Базовая ошибка конвейера импорта."""


class ConfigError(IngestError):
    """Настройки невалидны."""


class DocumentError(IngestError):
    """Проблема с конкретным документом."""

    def __init__(self, path, reason: str) -> None:
        super().__init__(f"{path}: {reason}")
        self.path = path
        self.reason = reason


for exc in (ConfigError("нет каталога"), DocumentError("a.json", "битый JSON")):
    try:
        raise exc
    except IngestError as e:            # ловит обоих наследников
        print(type(e).__name__, "->", e)

ConfigError -> нет каталога
DocumentError -> a.json: битый JSON


### Задание 5. LBYL vs EAFP

Реализуйте чтение файла двумя способами. Объясните, в каком случае EAFP надёжнее.

In [9]:
import os

path = "no_such_file.txt"

# LBYL
if os.path.exists(path):
    with open(path, encoding="utf-8") as f:
        data = f.read()
else:
    print("LBYL: файла нет")

# EAFP
try:
    with open(path, encoding="utf-8") as f:
        data = f.read()
except FileNotFoundError:
    print("EAFP: файла нет")

LBYL: файла нет
EAFP: файла нет


### Задание 6. Файловые режимы

Создайте файл в режиме `'w'`, допишите в режиме `'a'`, прочитайте. Объясните, что произойдёт при повторном открытии в `'w'`.

In [11]:
with open("test.txt", "w", encoding="utf-8") as f:
    f.write("Строка 1\nСтрока 2\nСтрока 3\n")

with open("test.txt", "a", encoding="utf-8") as f:
    f.write("Строка 4\n")

with open("test.txt", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        print(i, line.rstrip())

1 Строка 1
2 Строка 2
3 Строка 3
4 Строка 4


### Задание 7. JSON: сериализация и десериализация

Сохраните словарь с кириллицей через `json.dump` с `ensure_ascii=False`, прочитайте обратно. Поймайте `json.JSONDecodeError` на битом JSON.

In [12]:
import json

data = {"город": "Москва", "население": 12_500_000, "районы": ["ЦАО", "САО"]}

with open("city.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

with open("city.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
print("Прочитано:", loaded)

try:
    json.loads('{"a": 1, "b": }')
except json.JSONDecodeError as exc:
    print("Ошибка JSON:", exc)

Прочитано: {'город': 'Москва', 'население': 12500000, 'районы': ['ЦАО', 'САО']}
Ошибка JSON: Expecting value: line 1 column 15 (char 14)


### Задание 8. Работа с `pathlib`

Создайте дерево каталогов, запишите два JSON-файла, обойдите все `.json` через `rglob`, выведите имя, размер и полный путь для каждого.

In [14]:
from pathlib import Path

Path("tree/raw").mkdir(parents=True, exist_ok=True)
Path("tree/processed").mkdir(parents=True, exist_ok=True)

Path("tree/raw/a.json").write_text('{"a": 1}', encoding="utf-8")
Path("tree/raw/b.json").write_text('{"b": 2}', encoding="utf-8")

for p in Path("tree").rglob("*.json"):
    print(f"{p.name:10s} {p.stat().st_size:>5} B   {p.absolute()}")

a.json         8 B   d:\maestria\I semestre\python-labs-gutierrez\lab3\tree\raw\a.json
b.json         8 B   d:\maestria\I semestre\python-labs-gutierrez\lab3\tree\raw\b.json


### Задание 9. Свой модуль

Создайте файл `jsonutils.py` с функциями `read_json` и `write_json`, блоком `if __name__ == "__main__"`. Импортируйте и используйте его в ноутбуке.

In [15]:
%%writefile jsonutils.py
"""Утилиты для работы с JSON."""
import json
from pathlib import Path


def read_json(path) -> dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(data: dict, path) -> None:
    Path(path).write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


if __name__ == "__main__":
    write_json({"hello": "мир"}, "demo_module.json")
    print(read_json("demo_module.json"))

Writing jsonutils.py


In [16]:
import jsonutils
jsonutils.write_json({"a": 1, "b": "текст"}, "module_test.json")
print(jsonutils.read_json("module_test.json"))

{'a': 1, 'b': 'текст'}


### Задание 10. Логирование с уровнями

Настройте логгер с двумя handler'ами (консоль INFO, файл DEBUG с ротацией). Выведите все пять уровней. Проверьте содержимое файла.

In [17]:
import logging
from logging.handlers import RotatingFileHandler

logger = logging.getLogger("task10")
logger.setLevel(logging.DEBUG)

console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

fh = RotatingFileHandler("task10.log", maxBytes=100_000, backupCount=3, encoding="utf-8")
fh.setLevel(logging.DEBUG)
fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s"))

logger.addHandler(console)
logger.addHandler(fh)

for level in (logging.DEBUG, logging.INFO, logging.WARNING,
              logging.ERROR, logging.CRITICAL):
    logger.log(level, logging.getLevelName(level))

print("\ntask10.log:")
print(open("task10.log", encoding="utf-8").read())

DEBUG task10: DEBUG
INFO: INFO
INFO task10: INFO
WARNING task10: WARNING
ERROR: ERROR
ERROR task10: ERROR
CRITICAL: CRITICAL
CRITICAL task10: CRITICAL



task10.log:
2026-10-21 17:37:23,285 [DEBUG] task10: DEBUG
2026-10-21 17:37:23,288 [INFO] task10: INFO
2026-10-21 17:37:23,291 [WARNING] task10: WARNING
2026-10-21 17:37:23,292 [ERROR] task10: ERROR
2026-10-21 17:37:23,293 [CRITICAL] task10: CRITICAL



In [19]:
import shutil
from pathlib import Path

shutil.rmtree("inbox", ignore_errors=True)
Path("inbox/good").mkdir(parents=True)
Path("inbox/nested").mkdir(parents=True)

Path("inbox/good/a.json").write_text(
    '{"id": 1, "name": "Alice", "age": 30}', encoding="utf-8"
)
Path("inbox/good/b.json").write_text(
    '{"id": 2, "name": "Bob", "age": 25}', encoding="utf-8"
)
Path("inbox/nested/c.json").write_text(
    '{"id": 3, "name": "Carol", "age": 28}', encoding="utf-8"
)
Path("inbox/broken.json").write_text(
    '{"id": 4, "name": }', encoding="utf-8"
)
Path("inbox/no_keys.json").write_text(
    '{"foo": "bar"}', encoding="utf-8"
)
Path("inbox/not_object.json").write_text(
    '[1, 2, 3]', encoding="utf-8"
)

print(f"Всего файлов: {len(list(Path('inbox').rglob('*.json')))}")

Всего файлов: 6


### Задание 11. Настройки из окружения

Класс `Settings` с полями `source_dir`, `output_path`, `required_keys`. Метод `from_env()` читает окружение и валидирует.

In [21]:
import os
from pathlib import Path


class ConfigError(Exception):
    """Настройки невалидны."""


class Settings:
    def __init__(self, source_dir: Path, output_path: Path,
                 required_keys: tuple[str, ...]) -> None:
        self.source_dir = source_dir
        self.output_path = output_path
        self.required_keys = required_keys

    @classmethod
    def from_env(cls) -> "Settings":
        source = Path(os.environ.get("SOURCE_DIR", "inbox"))
        output = Path(os.environ.get("OUTPUT_PATH", "catalog.json"))
        keys = tuple(k.strip() for k in os.environ.get("REQUIRED_KEYS", "id,name").split(",") if k.strip())

        if not source.exists():
            raise ConfigError(f"Каталог не найден: {source}")
        if not source.is_dir():
            raise ConfigError(f"Ожидался каталог: {source}")
        if not keys:
            raise ConfigError("REQUIRED_KEYS пустой")

        return cls(source, output, keys)


os.environ["SOURCE_DIR"] = "inbox"
os.environ["REQUIRED_KEYS"] = "id,name"
settings = Settings.from_env()
print(settings.source_dir, settings.output_path, settings.required_keys)

inbox catalog.json ('id', 'name')


### Задание 12. Обход каталога

Метод `Ingester.iter_files()` — рекурсивный обход через `Path.rglob`, отсортированный итератор.

In [22]:
class Ingester:
    def __init__(self, settings: Settings) -> None:
        self.settings = settings
        self.errors: list[str] = []

    def iter_files(self):
        """Все .json в source_dir, рекурсивно, отсортированные."""
        return iter(sorted(self.settings.source_dir.rglob("*.json")))


ing = Ingester(settings)
for p in ing.iter_files():
    print(p)

inbox\broken.json
inbox\good\a.json
inbox\good\b.json
inbox\nested\c.json
inbox\no_keys.json
inbox\not_object.json


### Задание 13. Чтение одного документа

Метод `load_document(path) -> dict` с полной валидацией. При проблеме — `DocumentError` с причиной. Ошибки `OSError` / `JSONDecodeError` оборачиваются через `raise ... from exc`.

In [23]:
import json


class DocumentError(Exception):
    def __init__(self, path: Path, reason: str) -> None:
        super().__init__(f"{path.name}: {reason}")
        self.path = path
        self.reason = reason


class IngesterWithLoad(Ingester):
    def load_document(self, path: Path) -> dict:
        try:
            text = path.read_text(encoding="utf-8")
        except OSError as exc:
            raise DocumentError(path, f"не читается ({exc.strerror})") from exc

        try:
            data = json.loads(text)
        except json.JSONDecodeError as exc:
            raise DocumentError(path, f"битый JSON ({exc.msg})") from exc

        if not isinstance(data, dict):
            raise DocumentError(path, f"ожидался объект, получено {type(data).__name__}")

        missing = [k for k in self.settings.required_keys if k not in data]
        if missing:
            raise DocumentError(path, f"нет полей: {', '.join(missing)}")

        return data


ing = IngesterWithLoad(settings)
for p in ing.iter_files():
    try:
        print(f"OK    {p.name}: {ing.load_document(p)}")
    except DocumentError as exc:
        print(f"FAIL  {exc}")

FAIL  broken.json: битый JSON (Expecting value)
OK    a.json: {'id': 1, 'name': 'Alice', 'age': 30}
OK    b.json: {'id': 2, 'name': 'Bob', 'age': 25}
OK    c.json: {'id': 3, 'name': 'Carol', 'age': 28}
FAIL  no_keys.json: нет полей: id, name
FAIL  not_object.json: ожидался объект, получено list


### Задание 14. Конвейер с изоляцией ошибок

Метод `run() -> dict` — обходит файлы, изолирует ошибки, формирует отчёт.

In [24]:
import logging

logger = logging.getLogger("ingest")


class IngesterRun(IngesterWithLoad):
    def run(self) -> dict:
        files = list(self.iter_files())
        records: list[dict] = []
        skipped = 0

        for path in files:
            try:
                records.append(self.load_document(path))
            except DocumentError as exc:
                skipped += 1
                self.errors.append(str(exc))
                logger.warning("Пропущен %s", exc)
            except Exception:
                skipped += 1
                logger.exception("Неожиданная ошибка на %s", path)

        return {
            "source": str(self.settings.source_dir),
            "total_files": len(files),
            "imported": len(records),
            "skipped": skipped,
            "errors": self.errors,
            "records": records,
        }


report = IngesterRun(settings).run()
for k, v in report.items():
    if k != "records":
        print(f"{k}: {v}")
print("records:", report["records"])

WARNING ingest: Пропущен broken.json: битый JSON (Expecting value)
WARNING ingest: Пропущен no_keys.json: нет полей: id, name
WARNING ingest: Пропущен not_object.json: ожидался объект, получено list


source: inbox
total_files: 6
imported: 3
skipped: 3
errors: ['broken.json: битый JSON (Expecting value)', 'no_keys.json: нет полей: id, name', 'not_object.json: ожидался объект, получено list']
records: [{'id': 1, 'name': 'Alice', 'age': 30}, {'id': 2, 'name': 'Bob', 'age': 25}, {'id': 3, 'name': 'Carol', 'age': 28}]
